# Tomosynthesis (DBT) augmentation with MedAugmentX

Digital breast tomosynthesis is a reconstructed pseudo-3D volume, and it breaks
the assumptions most augmentation libraries are built on. Voxels are strongly
**anisotropic** — in-plane resolution is far finer than slice separation — and
the through-plane direction is a *reconstruction* axis, not a measured one.

Treating a DBT stack like an isotropic 3D volume produces anatomy that no
scanner could ever output. This notebook shows what to do instead.
### Before you start

Everything below runs on **synthetic phantoms** built from analytic shapes —
no patient data, no downloads, no network access. They are illustrations, not
validated physical models, so don't read clinical conclusions off them. The
transform strengths are deliberately exaggerated so the effect is visible in a
single figure; they are *not* recommended training policies.

Install what this notebook needs:

```bash
pip install "medaugmentx[notebooks]"
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import medaugmentx

print("MedAugmentX", medaugmentx.__version__)


def show(*panels, limits=None, cmap="gray", title=None):
    """Display `(label, volume)` pairs side by side on one fixed grey scale.

    A shared scale matters: normalising each panel independently would hide
    exactly the intensity shifts these transforms are meant to introduce.
    """
    fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.4))
    axes = np.atleast_1d(axes)
    planes = []
    for _, volume in panels:
        image = volume.image if hasattr(volume, "image") else volume
        planes.append(image if image.ndim == 2 else image[image.shape[0] // 2])
    lo, hi = limits if limits else (min(p.min() for p in planes), max(p.max() for p in planes))
    for ax, (label, _), plane in zip(axes, panels, planes):
        ax.imshow(plane, cmap=cmap, vmin=lo, vmax=hi, interpolation="nearest")
        ax.set_title(label, fontsize=11)
        ax.set_axis_off()
    if title:
        fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()


## 1. An anisotropic 3D volume

Look at `spacing`: 1.0 mm between slices but 0.25 mm in-plane — a 4:1 ratio.
Every transform below reads that spacing rather than assuming cubic voxels.


In [ ]:
from medaugmentx.phantoms import dbt_phantom

volume = dbt_phantom()
print("shape   ", volume.shape, " (z, y, x)")
print("spacing ", volume.spacing, " <- 4x anisotropy between z and x")
print("is_3d   ", volume.is_3d)

mid_z, mid_y = volume.shape[0] // 2, volume.shape[1] // 2
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
axes[0].imshow(volume.image[mid_z], cmap="gray", vmin=0, vmax=0.8, interpolation="nearest")
axes[0].set_title(f"In-plane slice z={mid_z}", fontsize=11)
axes[1].imshow(volume.image[:, mid_y, :], cmap="gray", vmin=0, vmax=0.8,
               interpolation="nearest", aspect="auto")
axes[1].set_title(f"Depth view y={mid_y}  (z down, x across)", fontsize=11)
for ax in axes:
    ax.set_axis_off()
fig.tight_layout()
plt.show()


def depth_view(vol):
    """The (z, x) plane through the middle y index — where DBT artifacts show."""
    return vol.image[:, vol.shape[1] // 2, :]


def show_depth(*panels, limits=(0, 0.8)):
    fig, axes = plt.subplots(len(panels), 1, figsize=(11, 2.6 * len(panels)))
    for ax, (label, vol) in zip(np.atleast_1d(axes), panels):
        ax.imshow(depth_view(vol), cmap="gray", vmin=limits[0], vmax=limits[1],
                  interpolation="nearest", aspect="auto")
        ax.set_title(label, fontsize=11)
        ax.set_axis_off()
    fig.tight_layout()
    plt.show()

## 2. Slab shift — reconstruction centre variation

The same patient imaged twice will not have the reconstruction planes land at
the same depths. `SlabShift` models that z-offset. It is a small, cheap
transform that buys real cross-study robustness.


In [ ]:
from medaugmentx.transforms import SlabShift

shifted = SlabShift(max_shift=3, seed=7)(volume)
show_depth(("Original (depth view)", volume), ("SlabShift(max_shift=3)", shifted))

## 3. Limited-angle blur — the defining DBT artifact

DBT reconstructs from a narrow arc (typically 15–50°), not a full rotation. The
consequence is severe: **in-plane resolution is preserved, through-plane
resolution is not**. Structures smear along z, and the narrower the arc, the
worse it gets.

`LimitedAngleBlur` applies exactly this anisotropic blur. Compare a 15° arc
against a 45° one.


In [ ]:
from medaugmentx.transforms import LimitedAngleBlur

narrow = LimitedAngleBlur(arc_degrees=15.0, base_sigma=1.8, seed=7)(volume)
wide = LimitedAngleBlur(arc_degrees=45.0, base_sigma=1.8, seed=7)(volume)

show_depth(("Original", volume),
           ("15 deg arc  (more through-plane smear)", narrow),
           ("45 deg arc  (less)", wide))

# In-plane detail survives; the depth axis is what degrades.
print("in-plane  std, original vs 15 deg:"
      f" {volume.image[mid_z].std():.4f} vs {narrow.image[mid_z].std():.4f}")
print("depth     std, original vs 15 deg:"
      f" {depth_view(volume).std():.4f} vs {depth_view(narrow).std():.4f}")

## 4. Slice dropout and compression variation

- **`SliceDropout`** — blanks reconstructed planes, modelling dropouts and
  teaching a model not to depend on any single slice being present.
- **`CompressionVariation`** — the breast is compressed differently at each
  visit, so anatomy is stretched along one axis between studies.


In [ ]:
from medaugmentx.transforms import CompressionVariation, SliceDropout

dropped = SliceDropout(num_slices=3, seed=7)(volume)
squeezed = CompressionVariation(scale=(0.85, 0.9), axis="y", seed=7)(volume)

show_depth(("Original", volume), ("SliceDropout(num_slices=3)", dropped))

# Compression rescales anatomy along one axis but keeps the array shape, so the
# volume still batches with its neighbours.
print("compression kept the array shape:", squeezed.shape == volume.shape)
show(("Original, in-plane", volume), ("CompressionVariation, in-plane", squeezed),
     limits=(0, 0.8))

## 5. Anisotropic elastic deformation

An isotropic elastic warp would deform z as freely as x — physically wrong for
DBT. `AnisotropicElastic` takes per-axis `alpha` (displacement magnitude) and
`sigma` (smoothness), so you can deform in-plane while barely touching depth.

The defaults already encode that asymmetry: `alpha=(100, 100, 8)`.


In [ ]:
from medaugmentx.transforms import AnisotropicElastic

warped = AnisotropicElastic(alpha=(120.0, 120.0, 6.0), sigma=(9.0, 9.0, 2.0), seed=7)(volume)
show_depth(("Original", volume), ("AnisotropicElastic (in-plane >> depth)", warped))

## 6. The DBT preset, and cost

`dbt_pipeline()` composes these. 3D volumes are large, so time your pipeline
before putting it in a data loader — `benchmarks/benchmark.py` reports
per-transform cost if you need to find the expensive step.


In [ ]:
import time

from medaugmentx import pipeline_summary
from medaugmentx.presets import dbt_pipeline

pipeline = dbt_pipeline(seed=0)
print(pipeline_summary(pipeline))

start = time.perf_counter()
augmented = pipeline(volume)
print(f"\n{volume.shape} volume augmented in {time.perf_counter() - start:.3f} s")

show_depth(("Original", volume), ("dbt_pipeline(seed=0)", augmented))

## Where to go next

- **[`docs/API_REFERENCE.md`](../docs/API_REFERENCE.md)** — every public import.
- **[`docs/RESEARCH_GUIDE.md`](../docs/RESEARCH_GUIDE.md)** — replay, worker
  seeds, and how to report augmentation in a paper.
- **[`examples/`](../examples/)** — runnable scripts, including
  `safe_augmentation.py` (validator + `Guard`) and `keypoints_bboxes.py`.
- **The other tutorials** — `01_mri_augmentation.ipynb`,
  `02_ct_augmentation.ipynb`, `03_dbt_augmentation.ipynb`.

Choosing augmentation strength is an empirical question for *your* dataset and
task. Start weak, look at the images, and validate against a held-out set.
